# Secondary Demand Forecasting — Training

Model: **3-model voting ensemble** — CatBoost + XGBoost + LightGBM (all MAE loss, log1p target).
Uses **lag features** of `secondary` plus rolling stats, `tertiary_lag_1`, and engineered concurrent features.
No `growth_1` leakage. Saves `../models/secondary_model.joblib`.


In [ ]:
import pandas as pd, numpy as np, math, joblib
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
import xgboost as xgb
from sklearn.ensemble import VotingRegressor
from sklearn.metrics import r2_score
import warnings; warnings.filterwarnings('ignore')


### 1. Feature engineering (lags + rolling + cross-channel)


In [ ]:
TARGET='secondary'
FEATS=['ret_stock','ret_dos','wod','activating_outlet','dbr_stock','dbr_dos','stocking_outlet','tertiary',
       'seasonality_1(sin)','seasonality_2(cos)','total_stock','activation_ratio','tertiary_lag_1',
       'secondary_lag_1','secondary_lag_2','secondary_lag_3','secondary_lag_4','secondary_lag_6','secondary_lag_8',
       'secondary_roll_mean_3','secondary_roll_mean_4','secondary_roll_mean_8','secondary_roll_std_4','secondary_roll_max_4','secondary_diff_lag']
LOGF=['ret_stock','ret_dos','wod','activating_outlet','dbr_stock','dbr_dos','stocking_outlet','tertiary','total_stock',
      'tertiary_lag_1','secondary_lag_1','secondary_lag_2','secondary_lag_3','secondary_lag_4','secondary_lag_6','secondary_lag_8',
      'secondary_roll_mean_3','secondary_roll_mean_4','secondary_roll_mean_8','secondary_roll_max_4']

def seasonality(df):
    o=df.copy(); a=2*math.pi*pd.to_numeric(o['week'],errors='coerce').fillna(0)/59.0
    o['seasonality_1(sin)']=np.sin(a); o['seasonality_2(cos)']=np.cos(a); return o

def add_features(df):
    o=df.copy()
    sk=['model_family']+(['model_code'] if 'model_code' in o else [])+(['year'] if 'year' in o else [])+(['_source_rank'] if '_source_rank' in o else [])+['week']
    o=o.sort_values(sk).reset_index(drop=True)
    g=o.groupby(['model_family'],sort=False)[TARGET]
    for l in (1,2,3,4,6,8): o[f'secondary_lag_{l}']=g.shift(l)
    sh=g.shift(1)
    for w in (3,4,8): o[f'secondary_roll_mean_{w}']=sh.rolling(w,min_periods=w).mean()
    o['secondary_roll_std_4']=sh.rolling(4,min_periods=4).std()
    o['secondary_roll_max_4']=sh.rolling(4,min_periods=4).max()
    o['secondary_diff_lag']=o['secondary_lag_1']-o['secondary_lag_2']
    o['tertiary_lag_1']=o.groupby(['model_family'],sort=False)['tertiary'].shift(1)
    o['total_stock']=pd.to_numeric(o['ret_stock'],errors='coerce').fillna(0)+pd.to_numeric(o['dbr_stock'],errors='coerce').fillna(0)
    o['activation_ratio']=pd.to_numeric(o['activating_outlet'],errors='coerce').fillna(0)/(pd.to_numeric(o['stocking_outlet'],errors='coerce').fillna(0)+1e-5)
    return o

def build_X(df):
    o=df.copy()
    for c in FEATS:
        if c not in o: o[c]=0.0
    X=o[FEATS].apply(pd.to_numeric,errors='coerce').fillna(0)
    for c in LOGF:
        if c in X: X[c]=np.log1p(np.clip(X[c],0,None))
    return X.replace([np.inf,-np.inf],np.nan).fillna(0)


### 2. Train ensemble on cleaned_secondary_final.csv


In [ ]:
csv=pd.read_csv('cleaned_secondary_final.csv')
csv['start_date']=pd.to_datetime(csv['start_date']); csv['year']=csv['start_date'].dt.year
csv=csv[csv[TARGET]>=0].copy()
csv=add_features(seasonality(csv)).dropna(subset=[f'secondary_lag_{l}' for l in (1,2,3,4,6,8)]).reset_index(drop=True)
X=build_X(csv); y=np.log1p(np.clip(csv[TARGET].values,0,None))

cb=CatBoostRegressor(iterations=800,learning_rate=0.05,depth=6,loss_function='MAE',random_seed=42,verbose=0,allow_writing_files=False)
xg=xgb.XGBRegressor(objective='reg:absoluteerror',n_estimators=600,max_depth=5,learning_rate=0.05,subsample=0.85,colsample_bytree=0.85,random_state=42,n_jobs=-1)
lg=LGBMRegressor(objective='mae',n_estimators=600,learning_rate=0.05,max_depth=6,num_leaves=31,subsample=0.85,colsample_bytree=0.85,random_state=42,verbose=-1)
model=VotingRegressor([('cat',cb),('xgb',xg),('lgb',lg)])
model.fit(X,y)
print('Trained on', len(csv), 'rows')


### 3. Evaluate (history weeks 41-48 + test weeks 49-59)


In [ ]:
def acc(a,b):
    a=np.asarray(a); b=np.clip(np.asarray(b),0,None)
    return max(0,100-np.mean(np.abs((a-b)/np.clip(np.abs(a),1,None))*100))

h=pd.read_excel('HEROSHAK2025H11-Secondary-Lag.xlsx'); h['_source_rank']=0
t=pd.read_excel('HEROSHAK2025H11-Secondary-test.xlsx'); t['_source_rank']=1
comb=pd.concat([h,t],ignore_index=True); comb['_is_test']=comb['_source_rank'].eq(1)
comb=add_features(seasonality(comb)).dropna(subset=[f'secondary_lag_{l}' for l in (1,2,3,4,6,8)]).reset_index(drop=True)
sc=comb[comb['_is_test']].copy(); yt=sc[TARGET].values; wk=sc['week'].values
pred=np.clip(np.expm1(model.predict(build_X(sc))),0,None)
print(f'Accuracy : {acc(yt,pred):.2f}%  R2={r2_score(yt,pred):.3f}')
display(pd.DataFrame({'Week':wk,'Actual':yt,'Pred':np.round(pred,0),'APE%':np.round(np.abs(yt-pred)/np.clip(yt,1,None)*100,1)}))


### 4. Save model artifact


In [ ]:
import os; os.makedirs('../models',exist_ok=True)
joblib.dump({'model':model,'features':FEATS,'log_feature_cols':LOGF,'target':'secondary',
             'target_log1p':True,'scaler':None,'model_version':'Secondary_Ensemble_v1'},
            '../models/secondary_model.joblib')
print('Saved ../models/secondary_model.joblib')
